In [ ]:
from build_decision_dataset import (
    build_decision_dataset,
    summarize_decision_dataset,
)

dataset = build_decision_dataset(
    n=10,
    n_instances=20,
    seed_offset=0,
    time_limit_s=3.0,
    include_return_to_depot=True,
    verbose=True,
)

summary = summarize_decision_dataset(dataset)
print(summary)
print("Nb exemples :", len(dataset))

## Génération des datasets oracle (n=10, 20, 50)

In [ ]:
from build_decision_dataset import build_decision_dataset, save_decision_dataset, summarize_decision_dataset

CONFIGS = [
    dict(n=50, n_instances=1000, time_limit_s=5.0, output="decision_dataset_n50.npy"),
]

for cfg in CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Génération dataset n={cfg['n']} ({cfg['n_instances']} instances)")
    print(f"{'='*60}")
    dataset = build_decision_dataset(
        n=cfg["n"],
        n_instances=cfg["n_instances"],
        seed_offset=0,
        time_limit_s=cfg["time_limit_s"],
        include_return_to_depot=False,
        verbose=True,
    )
    save_decision_dataset(dataset, cfg["output"])
    summary = summarize_decision_dataset(dataset)
    print(f"  → {summary['n_examples']} décisions, {summary['n_instances']} instances valides")

print("\n✓ Tous les datasets générés.")

In [ ]:
ex = dataset[0]

print("=== Exemple 0 ===")
print("seed              :", ex["seed"])
print("current_node      :", ex["current_node"])
print("target_next_node  :", ex["target_next_node"])
print("current_time      :", ex["current_time"])
print("current_load      :", ex["current_load"])
print("vehicle_id        :", ex["vehicle_id"])
print("step_in_vehicle   :", ex["step_in_vehicle"])
print("route_id          :", ex["route_id"])

print("\nShapes")
print("node_features     :", ex["node_features"].shape)
print("action_features   :", ex["action_features"].shape)
print("served_mask       :", ex["served_mask"].shape)
print("remaining_mask    :", ex["remaining_mask"].shape)
print("feasible_mask     :", ex["feasible_mask"].shape)
print("dynamic_global    :", ex["dynamic_global"].shape)

print("\nMasks")
print("served_mask       :", ex["served_mask"])
print("feasible_mask     :", ex["feasible_mask"])

In [ ]:
bad = 0
for i, ex in enumerate(dataset):
    target = ex["target_next_node"]
    if ex["feasible_mask"][target] != 1.0:
        bad += 1
        print("Exemple incohérent :", i, "target=", target)

print("Nb incohérences :", bad)

In [ ]:
import numpy as np

targets = np.array([ex["target_next_node"] for ex in dataset])
print("Taux cible = dépôt :", np.mean(targets == 0))

In [ ]:
from build_decision_dataset import save_decision_dataset

save_decision_dataset(dataset, "decision_dataset_n10.npy")

In [ ]:
from build_decision_dataset import load_decision_dataset

dataset_loaded = load_decision_dataset("decision_dataset_n10.npy")
print("Nb exemples rechargés :", len(dataset_loaded))
print("Premier target :", dataset_loaded[0]["target_next_node"])

In [ ]:
import torch
from model import VRPTWDecisionModel
from build_decision_dataset import load_decision_dataset

dataset = load_decision_dataset("decision_dataset_n10.npy")

model = VRPTWDecisionModel()

# prendre un exemple
ex = dataset[0]

# convertir en tensors
node_features = torch.tensor(ex["node_features"], dtype=torch.float32)
action_features = torch.tensor(ex["action_features"], dtype=torch.float32)
dynamic_global = torch.tensor(ex["dynamic_global"], dtype=torch.float32)
served_mask = torch.tensor(ex["served_mask"], dtype=torch.float32)
feasible_mask = torch.tensor(ex["feasible_mask"], dtype=torch.float32)
current_node = ex["current_node"]

# forward
logits = model(
    node_features,
    action_features,
    current_node,
    dynamic_global,
    served_mask,
    feasible_mask
)

print("Logits :", logits)
print("Action choisie :", torch.argmax(logits).item())
print("Cible oracle :", ex["target_next_node"])

In [ ]:
import torch
from model import VRPTWDecisionModel
from build_decision_dataset import load_decision_dataset

dataset = load_decision_dataset("decision_dataset_n10.npy")
model = VRPTWDecisionModel()

correct = 0
n = 50

for i in range(min(n, len(dataset))):
    ex = dataset[i]

    node_features = torch.tensor(ex["node_features"], dtype=torch.float32)
    action_features = torch.tensor(ex["action_features"], dtype=torch.float32)
    dynamic_global = torch.tensor(ex["dynamic_global"], dtype=torch.float32)
    served_mask = torch.tensor(ex["served_mask"], dtype=torch.float32)
    feasible_mask = torch.tensor(ex["feasible_mask"], dtype=torch.float32)
    current_node = ex["current_node"]

    logits = model(
        node_features,
        action_features,
        current_node,
        dynamic_global,
        served_mask,
        feasible_mask
    )

    pred = torch.argmax(logits).item()
    target = ex["target_next_node"]

    if pred == target:
        correct += 1

print(f"Accuracy brute avant entraînement : {correct}/{min(n, len(dataset))} = {correct/min(n, len(dataset)):.2%}")

In [ ]:
!python train_small.py --data decision_dataset_n10.npy --epochs 30 --batch_size 32 --lr 1e-3

In [ ]:
import json

with open("artifacts_train_small/history.json", "r", encoding="utf-8") as f:
    history = json.load(f)

history[:3], history[-3:]

In [ ]:
best = max(history, key=lambda x: x["val_acc"])
print(best)

In [ ]:
from preprocess import generate_instance
from inference_decoder import solve_with_heuristic, solve_with_model

instance = generate_instance(n=10, seed=42)

heur = solve_with_heuristic(instance)
mdl = solve_with_model(instance, "artifacts_train_small/best_model.pt")

print("=== Heuristique ===")
print("Routes :", heur["result"].routes)
print("Servis :", heur["result"].served_clients)
print("Non servis :", heur["result"].unserved_clients)
print("Coût :", round(heur["result"].total_cost, 2))
print("Faisable :", heur["checks"]["feasible"])
print("Erreurs :", heur["checks"]["errors"])

print("\n=== Modèle ===")
print("Routes :", mdl["result"].routes)
print("Servis :", mdl["result"].served_clients)
print("Non servis :", mdl["result"].unserved_clients)
print("Coût :", round(mdl["result"].total_cost, 2))
print("Faisable :", mdl["checks"]["feasible"])
print("Erreurs :", mdl["checks"]["errors"])

In [ ]:
from inference_decoder import benchmark_heuristic_vs_model

summary = benchmark_heuristic_vs_model(
    n=10,
    seeds=list(range(50)),
    checkpoint_path="artifacts_train_small/best_model.pt",
)

print("=== Résumé benchmark ===")
for k, v in summary.items():
    if k != "details":
        print(f"{k}: {v}")

In [ ]:
improved = []
worse = []

for row in summary["details"]:
    if (not row["heuristic_feasible"]) and row["model_feasible"]:
        improved.append(row)
    if row["heuristic_feasible"] and (not row["model_feasible"]):
        worse.append(row)

print("Nb améliorations nettes :", len(improved))
print("Nb régressions nettes   :", len(worse))

print("\nExemples d'amélioration :")
for row in improved[:5]:
    print(row)

## Résolution avec le modèle AM + Beam Search (W=20)

Approche inspirée de Kool et al. 2019 : au lieu de choisir le meilleur client à chaque étape (greedy), le beam search conserve les **20 meilleures solutions partielles** en parallèle et retourne celle avec le coût minimal.

Résultats sur n=10 : **gap ~6.5% vs oracle** (contre ~22% en greedy pur).

In [ ]:
import time
from preprocess import generate_instance
from decoder import decode_vrptw_beam_search, decode_vrptw_with_repair, verify_solution
from inference_decoder import load_trained_model, TorchModelScorer
from oracle import resoudre_instance

# ── Instance de test ──────────────────────────────────────────────────────────
SEED = 10042
N    = 10

instance = generate_instance(n=N, seed=SEED)
print(f"Instance n={N}, seed={SEED}")
print(f"  Capacité véhicule : {instance['capacity']:.0f}")
print(f"  Nombre de véhicules : {instance['n_vehicles']}")
print(f"  Horizon temporel : {instance['horizon']:.0f}")

# ── Chargement du modèle ──────────────────────────────────────────────────────
model  = load_trained_model("artifacts_train_small/best_model.pt")
scorer = TorchModelScorer(model=model)

# ── Résolution : Greedy ───────────────────────────────────────────────────────
t0 = time.time()
result_greedy = decode_vrptw_with_repair(instance, scorer=scorer)
t_greedy = time.time() - t0

# ── Résolution : Beam Search W=20 ─────────────────────────────────────────────
t0 = time.time()
result_beam = decode_vrptw_beam_search(instance, scorer=scorer, beam_width=20)
t_beam = time.time() - t0

# ── Résolution : Oracle OR-Tools (référence) ──────────────────────────────────
t0 = time.time()
sol_oracle = resoudre_instance(instance, time_limit_s=5.0)
t_oracle = time.time() - t0
oracle_cost = float(sol_oracle["cout"]) if sol_oracle and sol_oracle["faisable"] else None

# ── Affichage des résultats ───────────────────────────────────────────────────
def gap(cost, ref):
    return f"+{100*(cost-ref)/ref:.1f}%" if ref else "N/A"

print("\n" + "="*55)
print(f"{'Méthode':<22} {'Coût':>8} {'Gap oracle':>11} {'Temps':>8}")
print("-"*55)
if oracle_cost:
    print(f"{'Oracle (OR-Tools)':<22} {oracle_cost:>8.1f} {'—':>11} {t_oracle:>7.1f}s")
print(f"{'AM Greedy':<22} {result_greedy.total_cost:>8.1f} {gap(result_greedy.total_cost, oracle_cost):>11} {t_greedy*1000:>6.0f}ms")
print(f"{'AM Beam Search W=20':<22} {result_beam.total_cost:>8.1f} {gap(result_beam.total_cost, oracle_cost):>11} {t_beam*1000:>6.0f}ms")
print("="*55)

# ── Détail des routes (Beam Search) ──────────────────────────────────────────
checks = verify_solution(instance, result_beam)
print(f"\nSolution Beam Search W=20 — {'Faisable' if checks['feasible'] else 'INFAISABLE'}")
print(f"Clients servis : {result_beam.served_clients}")
if result_beam.unserved_clients:
    print(f"Non servis     : {result_beam.unserved_clients}")
print()
for i, route in enumerate(result_beam.routes):
    cost_r = sum(float(instance['dist'][route[k], route[k+1]]) for k in range(len(route)-1))
    print(f"  Véhicule {i+1} : {route}  (coût={cost_r:.1f})")